In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

The above allows us to start importing functions from core

In [ ]:
import os
import pandas as pd
import geopandas as gpd

from core.spatial_ops import (make_gdf_valid, remove_small_holes,
                         simplify_geometries, drop_duplicate_geometries,
                         export_layer)

# data source:
# https://github.com/microsoft/usbuildingfootprints
in_file_path = r'../data/input_data/colorado.geojson'

gdf = gpd.read_file(in_file_path)
gdf.to_crs(epsg=4326, inplace=True)
gdf.columns = gdf.columns.str.lower()

# review data
pd.set_option('display.max_columns', None)
gdf.head(5)

In [ ]:
keep_cols = ['release', 'geometry']
gdf = gdf[keep_cols].copy()

cols_rename_dict = {'geometry': 'geom'}
gdf.rename(columns=cols_rename_dict, inplace=True)
gdf.set_geometry('geom', inplace=True)

In [ ]:
gdf = drop_duplicate_geometries(gdf)
gdf = simplify_geometries(gdf, tolerance_meters=5)
gdf = remove_small_holes(gdf, min_hole_area_sqm=20)
gdf = make_gdf_valid(gdf)

In [ ]:
output_file_path = r'../data/output_data/colorado-buildings.pmtiles'
gdf.to_file(
    output_file_path,
    driver='PMTiles',
    MINZOOM=8,
    MAXZOOM=12,
)